In [2]:
import os
from google import genai
from dotenv import load_dotenv

# Load the API key securely from the .env file
load_dotenv()

# Initialize the modern Gemini client
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def classify_intent(customer_tweet):
    prompt = f"""
    You are an expert customer support routing agent for AppleSupport.
    Analyze the following customer tweet and classify it into exactly ONE of these intents:
    
    1. software_os_issue (iOS updates, app freezing, lagging, software bugs)
    2. hardware_power_issue (battery drain, phone died, cracked screen, won't turn on)
    3. account_login_issue (Apple ID, passwords, iCloud access)
    4. general_inquiry (asking questions, no bug mentioned)
    5. unknown (unclear, missing context, or gibberish)
    
    Respond ONLY with the exact name of the intent. Do not include any other text.
    
    Customer Tweet: "{customer_tweet}"
    Intent:
    """
    
    # Call the new 2.5 Flash model
        # Call the new 3.6 Flash model
    response = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=prompt
    )

    return response.text.strip()

# Let's test it on a real tweet!
test_tweet = "My iPhone 7 suddenly died and won't charge anymore."
predicted_intent = classify_intent(test_tweet)

print(f"Tweet: {test_tweet}")
print(f"AI Classification: {predicted_intent}")


Tweet: My iPhone 7 suddenly died and won't charge anymore.
AI Classification: hardware_power_issue


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# 1. Load our clean historical data
df = pd.read_csv("../data/processed/apple_support_conversations.csv")
historical_questions = df['text_customer'].fillna("").tolist()
historical_answers = df['text_brand'].tolist()

print("Building the search engine... (takes about 5 seconds)")
# 2. Build the TF-IDF Vectorizer (our search index)
vectorizer = TfidfVectorizer(stop_words='english')
# This converts all 106k customer questions into a mathematical matrix
tfidf_matrix = vectorizer.fit_transform(historical_questions) 

def retrieve_similar_answers(new_tweet, top_k=3):
    # Convert the new tweet into the same math format
    new_tweet_vector = vectorizer.transform([new_tweet])
    
    # Calculate how similar it is to every single historical question
    similarities = cosine_similarity(new_tweet_vector, tfidf_matrix).flatten()
    
    # Get the indices of the top matches
    top_indices = similarities.argsort()[-top_k:][::-1]
    
    results = []
    for idx in top_indices:
        results.append({
            "past_customer_question": historical_questions[idx],
            "past_apple_answer": historical_answers[idx],
            "similarity_score": round(similarities[idx], 3)
        })
    return results

# Let's test our Retriever!
test_tweet = "My iPhone 7 suddenly died and won't charge anymore."
print(f"\nSearching for similar past issues to: '{test_tweet}'\n")

top_results = retrieve_similar_answers(test_tweet, top_k=2)

for i, res in enumerate(top_results):
    print(f"--- MATCH {i+1} (Score: {res['similarity_score']}) ---")
    print(f"Past Question: {res['past_customer_question']}")
    print(f"How Apple Replied: {res['past_apple_answer']}\n")


Building the search engine... (takes about 5 seconds)

Searching for similar past issues to: 'My iPhone 7 suddenly died and won't charge anymore.'

--- MATCH 1 (Score: 0.52) ---
Past Question: My laptop just died out of no where and won’t turn back on or charge. Help  😭😭😭 @115858 @AppleSupport
How Apple Replied: @530928 Thank you for reaching out. Are you using an Apple certified charger? Also, have you tried moving to a different plug?

--- MATCH 2 (Score: 0.483) ---
Past Question: Hey @115858, my iPhone X died after being out of the box for less than an hour. Now it won’t charge or turn back on. What gives?
How Apple Replied: @784259 We're here for you! Have you tried any steps to get your iPhone powered on again, such as charging it or a force restart: https://t.co/OWRDHWSt4h

Also, which country are you located within?



In [4]:
import json

def handle_customer_tweet(new_tweet):
    # 1. RETRIEVE: Get top 2 historical examples from our TF-IDF search engine
    top_matches = retrieve_similar_answers(new_tweet, top_k=2)
    
    # 2. FORMAT CONTEXT: Turn them into a string we can feed to the LLM
    historical_context = ""
    for i, match in enumerate(top_matches):
        historical_context += f"Example {i+1}:\nCustomer: {match['past_customer_question']}\nAppleSupport: {match['past_apple_answer']}\n\n"
        
    # 3. GENERATE: The ultimate prompt combining Classification, Escalation, and Drafting
    prompt = f"""
    You are an expert AI customer support agent for AppleSupport.
    A customer just tweeted the following:
    "{new_tweet}"
    
    Here is how human AppleSupport agents have handled very similar tweets in the past:
    {historical_context}
    
    Your job is to output a JSON object with EXACTLY the following 4 keys:
    1. "intent": Classify the tweet into ONE of: [software_os_issue, hardware_power_issue, account_login_issue, general_inquiry, unknown].
    2. "action": Decide whether to "auto_handle" (if you can provide troubleshooting steps or an answer) or "escalate" (if it's 'unknown' or requires physical repair/checking their account).
    3. "reason": A 1-sentence reason for your action decision.
    4. "draft_reply": Write a reply to the customer. DO NOT hallucinate links. Adopt the exact tone and style of the historical AppleSupport examples provided above. Keep it under 280 characters (Twitter limit).
    
    Output purely valid JSON. No markdown formatting or extra text.
    """
    
    # Call Gemini
    response = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=prompt
    )
    
    # Parse the JSON response
    try:
        # Strip out any markdown code blocks (```json ... ```) just in case
        raw_text = response.text.replace("```json", "").replace("```", "").strip()
        agent_decision = json.loads(raw_text)
        return agent_decision
    except Exception as e:
        return {"error": "Failed to parse JSON", "raw_response": response.text}

# Let's test the full pipeline!
final_test = "My iPhone 7 suddenly died and won't charge anymore."
print(f"Incoming Tweet: {final_test}\n")

agent_result = handle_customer_tweet(final_test)

# Print it out nicely
print(json.dumps(agent_result, indent=4))


Incoming Tweet: My iPhone 7 suddenly died and won't charge anymore.

{
    "intent": "hardware_power_issue",
    "action": "auto_handle",
    "reason": "This issue can be initially addressed by offering standard power and charging troubleshooting steps.",
    "draft_reply": "We're here to help get your iPhone 7 back up and running. Have you tried using a different plug or charging cable, or attempted a force restart?"
}


In [5]:
import time

# 1. Load the golden set you labeled earlier
golden_df = pd.read_csv("../data/golden_eval_set.csv")

# For testing, let's just grab the first 3 rows you labeled
test_set = golden_df.head(3)

correct_intents = 0
total_tested = 0

print("Starting Evaluation...\n")

for index, row in test_set.iterrows():
    customer_tweet = row['text_customer']
    human_label = row['intent']  # This is the intent you typed by hand
    
    # 2. Run our AI Pipeline
    try:
        agent_result = handle_customer_tweet(customer_tweet)
        ai_intent = agent_result['intent']
        ai_reply = agent_result['draft_reply']
        
        # 3. Automated Metric: Accuracy
        is_correct = (ai_intent == human_label)
        if is_correct:
            correct_intents += 1
        total_tested += 1
        
        print(f"Tweet: {customer_tweet[:60]}...")
        print(f"Human Label: {human_label} | AI Label: {ai_intent}")
        print(f"AI Reply: {ai_reply}\n")
        
    except Exception as e:
        print(f"Error on row {index}: {e}")
        
    # Sleep for 2 seconds to avoid hitting free API rate limits!
    time.sleep(2) 

# Calculate Final Accuracy
accuracy = (correct_intents / total_tested) * 100
print(f"--- EVALUATION COMPLETE ---")
print(f"Intent Classification Accuracy: {accuracy:.1f}%")


Starting Evaluation...

Tweet: @AppleSupport How do I revert to previous iOS? Can I Delete ...
Human Label: software_os_issue | AI Label: software_os_issue
AI Reply: We'd be happy to help look into your options regarding your iOS version. Send us a DM to get started.

Tweet: Look y’all don’t own my I’s, okay!?@115858 I’m gonna need y’...
Human Label: general_inquiry | AI Label: software_os_issue
AI Reply: @115858 We recently released iOS 11.1.1 to fix this issue. Please back up your device and update via Settings > General > Software Update. How to back up: https://t.co/4f8hwT5to6

Tweet: @AppleSupport look, i just got this iPhone 8 and i would lik...
Human Label: hardware_power_issue | AI Label: hardware_power_issue
AI Reply: We want to make sure your iPhone is keeping up with you. Reach out to us in DM so we can take a closer look at your battery health and help get this resolved.

--- EVALUATION COMPLETE ---
Intent Classification Accuracy: 66.7%
